In [46]:
import pandas as pd
import random
import numpy as np
np.float_ = np.float64
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from danrerlib import mapping, utils
import pickle

data_path = '/Volumes/CSDG/Daniel_Gray/Daniel_scRNA_seq/Lawson_map_combined/all-sample/DGE_filtered/'
out_path = '/Volumes/CSDG/Daniel_Gray/Daniel_scRNA_seq/Pseudobulk/'

In [47]:
import warnings
warnings.simplefilter("ignore", FutureWarning)
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", RuntimeWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

In [48]:
## neural read in 
#adata = sc.read(data_path + 'neuralrelabelled_9_10_25.h5ad')
## all read in
#adata = sc.read(data_path + 'labelled.h5ad')

## DGE for one cell type

In [49]:
genotypes = ['S2b', 'Wik']
adata = adata[adata.obs["genotype"].isin(genotypes)].copy()

In [50]:
cell_subset = adata[adata[:, 'elavl3'].X == 0, :].copy()
#cell_subset = adata[adata[:, 'elavl3'].X > 0, :].copy()

/opt/anaconda3/envs/larvalheads/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3550: SparseEfficiencyWarning: Comparing a sparse matrix with 0 using == is inefficient, try using != instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [51]:
#example WITH pseudo replicates
pbs = []
for sample in cell_subset.obs['sample'].unique():
    samp_cell_subset = cell_subset[cell_subset.obs['sample'] == sample]
    
    samp_cell_subset.X = samp_cell_subset.layers['counts'] #make sure to use raw data
    
    
    
    indices = list(samp_cell_subset.obs_names)
    random.shuffle(indices)
    indices = np.array_split(np.array(indices), 2) #change number here for number of replicates deisred
    
    for i, pseudo_rep in enumerate(indices):
    
        rep_adata = sc.AnnData(X = samp_cell_subset[indices[i]].X.sum(axis = 0),
                               var = samp_cell_subset[indices[i]].var[[]])

        rep_adata.obs_names = [sample + '_' + str(i)]
        rep_adata.obs['sample'] = samp_cell_subset.obs['sample'].iloc[0]
        rep_adata.obs['replicate'] = i

        pbs.append(rep_adata)

In [52]:
pb = sc.concat(pbs)

In [53]:
pb.write_csvs(dirname= out_path+"/elavl3neg", skip_data=False)
#pb.write_csvs(dirname= out_path+"/elavl3pos", skip_data=False)

In [54]:
split = pb.obs['sample'].str.split('-', expand=True).rename(columns={0:'genotype', 1:'batch'})
pb.obs['genotype'] = split.genotype
pb.obs['batch'] = split.batch

In [55]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

In [56]:
counts = pd.DataFrame(pb.X, columns = pb.var_names) #need to do this to pass var names

In [57]:
dds = DeseqDataSet(
    counts = counts,
    metadata = pb.obs,
    design_factors="genotype")

In [58]:
sc.pp.filter_genes(dds, min_cells = 1)

In [59]:
dds.deseq2()

Fitting size factors...
... done in 0.01 seconds.

Fitting dispersions...
... done in 1.22 seconds.

Fitting dispersion trend curve...
... done in 0.31 seconds.

Fitting MAP dispersions...
... done in 1.51 seconds.

Fitting LFCs...
... done in 0.97 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.



In [60]:
dds

AnnData object with n_obs × n_vars = 8 × 29587
    obs: 'sample', 'replicate', 'genotype', 'batch'
    var: 'n_cells'
    uns: 'trend_coeffs', 'disp_function_type', '_squared_logres', 'prior_disp_var'
    obsm: 'design_matrix', 'size_factors', '_mu_LFC', '_hat_diagonals', 'replaceable'
    varm: '_normed_means', 'non_zero', '_MoM_dispersions', 'genewise_dispersions', '_genewise_converged', 'fitted_dispersions', 'MAP_dispersions', '_MAP_converged', 'dispersions', '_outlier_genes', 'LFC', '_LFC_converged', 'replaced', 'refitted', '_pvalue_cooks_outlier'
    layers: 'normed_counts', '_mu_hat', 'cooks'

In [61]:
# contrasting 
stat_res = DeseqStats(dds, contrast=('genotype', 'S2b', 'Wik'))
stat_res.summary()
de  = stat_res.results_df
de.sort_values('stat', ascending = False)
de.to_csv(out_path +'deg_elavl3neg_s2b.csv', index=True)
#de.to_csv(out_path +'deg_elavl3pos_s2b.csv', index=True)

Running Wald tests...
... done in 0.57 seconds.



Log2 fold change & Wald test p-value: genotype S2b vs Wik
                   baseMean  log2FoldChange     lfcSE      stat    pvalue  \
a1cf              73.096531        0.413628  1.175724  0.351807  0.724983   
a2ml              79.352191        1.099751  1.180599  0.931520  0.351585   
aaas               8.326546        0.090925  0.502139  0.181075  0.856308   
aacs             200.068809       -0.544628  0.231239 -2.355262  0.018510   
aadac              6.384104        0.125272  0.721080  0.173728  0.862079   
...                     ...             ...       ...       ...       ...   
LOC100534908       0.570228       -0.248427  2.324807 -0.106859  0.914901   
si:dkey-58b18.2    0.103290        1.032507  3.881409  0.266013  0.790229   
echdc1             1.921355        0.227049  1.035699  0.219223  0.826476   
sipa1l2-1         96.322910        0.133076  0.189651  0.701690  0.482873   
eme2               1.359630       -0.750199  1.451632 -0.516797  0.605298   

                 

In [62]:
with open(out_path +'dds_elavl3neg_s2b.pkl', "wb") as f: pickle.dump(dds, f)
#with open(out_path +'dds_elavl3pos_s2b.pkl', "wb") as f: pickle.dump(dds, f)

In [63]:
pd.value_counts(cell_subset.obs.genotype)

genotype
Wik    10877
S2b     8169
Name: count, dtype: int64

In [64]:
genotype = 'Wik'
cell_subset_wik = cell_subset[cell_subset.obs["genotype"]==(genotype)].copy()
# Sort out gene expression of cohesin genes
# Define your list of genes
gene_list = ["stag1a","stag1b","stag2a","stag2b","smc1al","smc1a","smc3","rad21a"]
# Subset the AnnData object for these genes
# Note: Ensure gene names match adata.var_names
adata_subset = cell_subset_wik[:, gene_list]
# Convert to DataFrame (cells as rows, genes as columns)
normalized_df = adata_subset.to_df()
normalized_df.to_csv(out_path+"cohesin_cells_"+genotype+"_elavlneg.csv")
## Do the same for protocadherins
# Define your list of genes
gene_list = ["pcdh1g1","pcdh1g2","pcdh1gb2","pcdh1g3","pcdh1g9","pcdh1gb9","pcdh1g11","pcdh1g18","pcdh1g22","pcdh1g22","pcdh1g26","pcdh1g29","pcdh1g30","pcdh1g31",
             "pcdh1g32","pcdh1g33","pcdh1gc5","pcdh1gc6"]
# Subset the AnnData object for these genes
# Note: Ensure gene names match adata.var_names
adata_subset = cell_subset_wik[:, gene_list]
# Convert to DataFrame (cells as rows, genes as columns)
normalized_df = adata_subset.to_df()
normalized_df.to_csv(out_path+"protocadherin_cells_"+genotype+"_elavlneg.csv")

In [65]:
genotype = 'S2b'
cell_subset_wik = cell_subset[cell_subset.obs["genotype"]==(genotype)].copy()
# Sort out gene expression of cohesin genes
# Define your list of genes
gene_list = ["stag1a","stag1b","stag2a","stag2b","smc1al","smc1a","smc3","rad21a"]
# Subset the AnnData object for these genes
# Note: Ensure gene names match adata.var_names
adata_subset = cell_subset_wik[:, gene_list]
# Convert to DataFrame (cells as rows, genes as columns)
normalized_df = adata_subset.to_df()
normalized_df.to_csv(out_path+"cohesin_cells_"+genotype+"_elavlneg.csv")
## Do the same for protocadherins
# Define your list of genes
gene_list = ["pcdh1g1","pcdh1g2","pcdh1gb2","pcdh1g3","pcdh1g9","pcdh1gb9","pcdh1g11","pcdh1g18","pcdh1g22","pcdh1g22","pcdh1g26","pcdh1g29","pcdh1g30","pcdh1g31",
             "pcdh1g32","pcdh1g33","pcdh1gc5","pcdh1gc6"]
# Subset the AnnData object for these genes
# Note: Ensure gene names match adata.var_names
adata_subset = cell_subset_wik[:, gene_list]
# Convert to DataFrame (cells as rows, genes as columns)
normalized_df = adata_subset.to_df()
normalized_df.to_csv(out_path+"protocadherin_cells_"+genotype+"_elavlneg.csv")

In [66]:
normalized_df

,pcdh1g1,pcdh1g2,pcdh1gb2,pcdh1g3,pcdh1g9,pcdh1gb9,pcdh1g11,pcdh1g18,pcdh1g22,pcdh1g22,pcdh1g26,pcdh1g29,pcdh1g30,pcdh1g31,pcdh1g32,pcdh1g33,pcdh1gc5,pcdh1gc6
01_02_36__s1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
01_02_87__s1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
01_05_40__s1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
01_06_71__s1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
01_07_08__s1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30_90_03__s8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
30_90_48__s8,1.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
30_91_63__s8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
30_91_88__s8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## DGE through a loop

In [67]:
#setup to contrast major cell types through loop
celltypes= adata.obs.CellTypeMajor.unique()
cont = ['genotype','R21','Wts']

In [68]:
# contrasting loop, change genotype and set cell types above then set names

for celltype in celltypes:

    cell_subset = adata[adata.obs["CellTypeMajor"] == celltype]
    pbs = []
    for sample in cell_subset.obs['sample'].unique():
        samp_cell_subset = cell_subset[cell_subset.obs['sample'] == sample]
        samp_cell_subset.X = samp_cell_subset.layers['counts'] #make sure to use raw data
        indices = list(samp_cell_subset.obs_names)
        random.shuffle(indices)
        indices = np.array_split(np.array(indices), 2) #change number here for number of replicates deisred
    
        for i, pseudo_rep in enumerate(indices):
            rep_adata = sc.AnnData(X = samp_cell_subset[indices[i]].X.sum(axis = 0),
                               var = samp_cell_subset[indices[i]].var[[]])
            rep_adata.obs_names = [sample + '_' + str(i)]
            rep_adata.obs['sample'] = samp_cell_subset.obs['sample'].iloc[0]
            rep_adata.obs['replicate'] = i

            pbs.append(rep_adata)
    pb = sc.concat(pbs)
    split = pb.obs['sample'].str.split('-', expand=True).rename(columns={0:'genotype', 1:'batch'})
    pb.obs['genotype'] = split.genotype
    pb.obs['batch'] = split.batch
    counts = pd.DataFrame(pb.X, columns = pb.var_names)
    dds = DeseqDataSet(
        counts = counts,
        metadata =pb.obs,
        design_factors="genotype")
    dds.deseq2()
    stat_res = DeseqStats(dds, contrast=(cont)) # set the genotype here
    stat_res.summary()
    de  = stat_res.results_df
    de.to_csv(out_path +'deg_' +celltype+'_r21.csv', index=True)

Fitting size factors...
... done in 0.01 seconds.

Fitting dispersions...
... done in 0.95 seconds.

Fitting dispersion trend curve...
... done in 0.25 seconds.

Fitting MAP dispersions...
... done in 1.21 seconds.

Fitting LFCs...
... done in 0.87 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.



KeyError: "The tested level ('R21') should correspond to one of the levels of 'genotype'"

In [ ]:
#setup to contrast Minor cell types of interest through loop
celltypes= ['Neuron','HB','Tectum','Pallium','FB-GABA','Torus Longitudinalis','Dorsal Habenula','Thalamus Proper',
           'Proliferative Glia','Perivent-layer','Neuron-Diff','MHBB-Diff','MB-Diff','Pallium-Diff','Serotonergic','Oligodendrocyte']
cont = ['genotype','S2b','Wik']

In [ ]:
# contrasting loop, change genotype and set cell types above then set names

for celltype in celltypes:

    cell_subset = adata[adata.obs["celltypefine"] == celltype]
    pbs = []
    for sample in cell_subset.obs['sample'].unique():
        samp_cell_subset = cell_subset[cell_subset.obs['sample'] == sample]
        samp_cell_subset.X = samp_cell_subset.layers['counts'] #make sure to use raw data
        indices = list(samp_cell_subset.obs_names)
        random.shuffle(indices)
        indices = np.array_split(np.array(indices), 2) #change number here for number of replicates deisred
    
        for i, pseudo_rep in enumerate(indices):
            rep_adata = sc.AnnData(X = samp_cell_subset[indices[i]].X.sum(axis = 0),
                               var = samp_cell_subset[indices[i]].var[[]])
            rep_adata.obs_names = [sample + '_' + str(i)]
            rep_adata.obs['sample'] = samp_cell_subset.obs['sample'].iloc[0]
            rep_adata.obs['replicate'] = i

            pbs.append(rep_adata)
    pb = sc.concat(pbs)
    split = pb.obs['sample'].str.split('-', expand=True).rename(columns={0:'genotype', 1:'batch'})
    pb.obs['genotype'] = split.genotype
    pb.obs['batch'] = split.batch
    counts = pd.DataFrame(pb.X, columns = pb.var_names)
    dds = DeseqDataSet(
        counts = counts,
        metadata =pb.obs,
        design_factors="genotype")
    dds.deseq2()
    stat_res = DeseqStats(dds, contrast=(cont)) # set the genotype here
    stat_res.summary()
    de  = stat_res.results_df
    de.to_csv(out_path +'deg_' +celltype+'_s2b.csv', index=True)

In [ ]:
#set genotypes for contrast
cont = ['genotype','R21','Wts']

In [ ]:
# pooling all cells for bulk

cell_subset = adata
pbs = []
for sample in cell_subset.obs['sample'].unique():
    samp_cell_subset = cell_subset[cell_subset.obs['sample'] == sample]
    samp_cell_subset.X = samp_cell_subset.layers['counts'] #make sure to use raw data
    indices = list(samp_cell_subset.obs_names)
    random.shuffle(indices)
    indices = np.array_split(np.array(indices), 5) #change number here for number of replicates deisred
    
    for i, pseudo_rep in enumerate(indices):
        rep_adata = sc.AnnData(X = samp_cell_subset[indices[i]].X.sum(axis = 0),
                               var = samp_cell_subset[indices[i]].var[[]])
        rep_adata.obs_names = [sample + '_' + str(i)]
        rep_adata.obs['sample'] = samp_cell_subset.obs['sample'].iloc[0]
        rep_adata.obs['replicate'] = i

        pbs.append(rep_adata)
pb = sc.concat(pbs)
split = pb.obs['sample'].str.split('-', expand=True).rename(columns={0:'genotype', 1:'batch'})
pb.obs['genotype'] = split.genotype
pb.obs['batch'] = split.batch
counts = pd.DataFrame(pb.X, columns = pb.var_names)
dds = DeseqDataSet(
    counts = counts,
    metadata =pb.obs,
    design_factors="genotype")
dds.deseq2()
stat_res = DeseqStats(dds, contrast=(cont)) # set the genotype here
stat_res.summary()
de  = stat_res.results_df
de.to_csv(out_path +'deg_All_r21.csv', index=True)

## Investigating DEGs

In [ ]:
sc.pl.umap(adata, color = ['CellType'], s = 1,legend_loc = 'on data',legend_fontsize=5)

In [ ]:
sc.pl.umap(adata, color = ['CellTypeMajor'], s = 1,legend_loc = 'on data',legend_fontsize=6)

In [ ]:
sc.pl.umap(adata, color = ['celltypefine'], s = 1,legend_loc = 'on data',legend_fontsize=6)

In [ ]:
gene_list = ['serpinc1','fgb','f5','f2']
# EXAMPLE
#gene_list = ['','','','']
#gene_list = ['','','','','','','','']
sc.pl.umap(adata, color=[i for i in gene_list], color_map='viridis', legend_fontsize=8, legend_loc = 'on data', s = 5)

In [ ]:
gene_list = ["pcdh1g1","pcdh1g2","pcdh1gb2","pcdh1g3","pcdh1g9","pcdh1gb9","pcdh1g11","pcdh1g18","pcdh1g22","pcdh1g22","pcdh1g26","pcdh1g29","pcdh1g30","pcdh1g31",
             "pcdh1g32","pcdh1g33","pcdh1gc5","pcdh1gc6"]
# EXAMPLE
#gene_list = ['','','','']
#gene_list = ['','','','','','','','']
sc.pl.umap(adata, color=[i for i in gene_list], color_map='turbo', legend_fontsize=8, legend_loc = 'on data', s = 5)

In [ ]:
gene_list = ["pcdh1g1","pcdh1a3"]
cell_subset_wik = cell_subset[cell_subset.obs["genotype"] == "Wik"]
sc.pl.umap(cell_subset_wik, color=[i for i in gene_list], color_map='turbo',vmax =3, legend_fontsize=10, s = 10, save = 'pcdh_wik.pdf')

cell_subset_s2b = cell_subset[cell_subset.obs["genotype"] == "S2b"]
sc.pl.umap(cell_subset_s2b, color=[i for i in gene_list], color_map='turbo',vmax =3, legend_fontsize=10, s = 10, save = 'pcdh_s2b.pdf')


In [ ]:
temp = adata.obs['CellType'] == "NSC"
temp = adata.obs.loc[temp, 'genotype']
temp.value_counts()

## Cell cycle

In [ ]:
gene_list = ['mad1l1','mad2l1','mad2l2','bub1bb','bub1','bub3']
# EXAMPLE
#gene_list = ['','','','']
#gene_list = ['','','','','','','','']
sc.pl.umap(adata, color=[i for i in gene_list], color_map='turbo', legend_fontsize=8, legend_loc = 'on data', s = 5)

In [ ]:
celltypes= ['NSC','Neuron-Diff','Ret-Prog','Cranial-Mes']

In [ ]:
cell_cycle_genes = [x.strip() for x in open('regev_lab_cell_cycle_genes.txt')]
cell_cycle_genes = [cell_cycle_genes.lower() for cell_cycle_genes in cell_cycle_genes]
print(cell_cycle_genes[:43])
print(cell_cycle_genes[43:])

In [ ]:
# directly turned caps to lowercase for zebrafish genes then manually switched to a for duplicated genes (**this is a lazy shortcut)
s_genes = ['mcm5', 'pcna', 'tyms', 'fen1', 'mcm2', 'mcm4', 'rrm1', 'unga','ungb', 'gins2', 'mcm6', 'cdca7a','cdca7b','dtl', 'prim1', 'uhrf1', 'hells',
           'rfc2', 'rpa2', 'nasp', 'rad51ap1', 'gmnn', 'wdr76', 'slbp', 'ccne2', 'ubr7', 'pold3', 'msh2', 'atad2', 'rad51', 'rrm2',
           'cdc45', 'cdc6', 'exo1', 'tipin', 'dscc1', 'blm', 'casp8ap2', 'usp1', 'pola1', 'chaf1b', 'brip1', 'e2f8']
           
g2m_genes = ['hmgb2a','hmgb2b', 'cdk1', 'nusap1', 'ube2c', 'birc5a','birc5b', 'tpx2', 'top2a', 'ndc80', 'cks2', 'nuf2', 'cks1b', 'mki67', 'tmpoa','tmpob', 'cenpf',
            'tacc3', 'smc4', 'ccnb2', 'ckap2l', 'aurkb', 'bub1', 'kif11', 'anp32e', 'tubb4b', 'gtse1', 'kif20ba','kif20bb', 'jpt1a', 'cdc20',
            'ttk', 'kif2c', 'rangap1a','rangap1b', 'ncapd2', 'dlgap5', 'cdca8', 'ect2', 'kif23', 'hmmr', 'aurka', 'anln', 'lbr', 'ckap5',
            'cenpe', 'ctcf', 'nek2', 'g2e3', 'gas2l3', 'cbx5']
sc.tl.score_genes_cell_cycle(adata, s_genes = s_genes, g2m_genes = g2m_genes)

In [ ]:
#cell_subset = adata[adata.obs["genotype"] == 'S2b']
cell_subset = adata[(adata.obs['genotype'] == 'S2b') & (adata.obs['CellType'] == 'NSC')]
#sc.tl.score_genes_cell_cycle(cell_subset, s_genes = s_genes, g2m_genes = g2m_genes)
v1 = cell_subset.obs[ 'S_score']
v2 = cell_subset.obs[ 'G2M_score' ]

mask = (v1 + v2 ) > -1
cell_subset = cell_subset[mask]
phase_colors = {
'G1': '#1E90FF',  # Blue
'G2M': '#00FF00', # Green
'S': '#FFA500'   # Orange
}
fig = plt.figure(figsize = (8,8));  c = 0
if 1: # Create Phase plot

    #phase_threshold = 0
    v1 = cell_subset.obs[ 'S_score']
    v2 = cell_subset.obs[ 'G2M_score' ]

    #c += 1; fig.add_subplot(1,n_x_subplots ,c)
    plt.title('Scanpy phase marks NSC S2b', fontsize = 20 )
    v_color =  cell_subset.obs[ 'phase']
    ax = sns.scatterplot(x=v1,y=v2, hue = v_color, palette = phase_colors, alpha =0.2 )
    # Changing fontsize for the legend: 
    plt.setp(ax.get_legend().get_texts(), fontsize=20) # for legend text
    plt.setp(ax.get_legend().get_title(), fontsize=20) # for legend title    
    plt.xlabel('Scanpy G1S' , fontsize = 20)
    plt.ylabel('Scanpy G2M', fontsize = 20 )
    plt.xlim(-0.1,0.30)
    plt.ylim(-0.1,0.55)
cell_subset.obs.phase.value_counts()

In [ ]:
#cell_subset = adata[adata.obs["genotype"] == 'Wik']
cell_subset = adata[(adata.obs['genotype'] == 'Wik') & (adata.obs['CellType'] == 'NSC')]
#sc.tl.score_genes_cell_cycle(cell_subset, s_genes = s_genes, g2m_genes = g2m_genes)
v1 = cell_subset.obs[ 'S_score']
v2 = cell_subset.obs[ 'G2M_score' ]

mask = (v1 + v2 ) > -1
cell_subset = cell_subset[mask]
phase_colors = {
'G1': '#1E90FF',  # Blue
'G2M': '#00FF00', # Green
'S': '#FFA500'   # Orange
}
fig = plt.figure(figsize = (8,8));  c = 0
if 1: # Create Phase plot

    #phase_threshold = 0
    v1 = cell_subset.obs[ 'S_score']
    v2 = cell_subset.obs[ 'G2M_score' ]

    #c += 1; fig.add_subplot(1,n_x_subplots ,c)
    plt.title('Scanpy phase marks NSC Wik', fontsize = 20 )
    v_color =  cell_subset.obs[ 'phase']
    ax = sns.scatterplot(x=v1,y=v2, hue = v_color, palette = phase_colors, alpha =0.2 )
    # Changing fontsize for the legend: 
    plt.setp(ax.get_legend().get_texts(), fontsize=20) # for legend text
    plt.setp(ax.get_legend().get_title(), fontsize=20) # for legend title    
    plt.xlabel('Scanpy G1S' , fontsize = 20)
    plt.ylabel('Scanpy G2M', fontsize = 20 )
    plt.xlim(-0.1,0.30)
    plt.ylim(-0.1,0.55)
cell_subset.obs.phase.value_counts()

In [ ]:
phasecounts = adata.obs.groupby(['CellType','genotype'])['phase'].value_counts().to_frame()
phasecounts = pd.DataFrame(phasecounts)
phasecounts = phasecounts.reset_index()
phasecounts = phasecounts.pivot_table(index=['CellType','genotype'], columns='phase', values='count')
phasecounts = phasecounts.reset_index()
phasecounts['sum'] = phasecounts.sum(axis = 1, numeric_only=True)
x = ['G1','G2M','S']
phasecounts[x] = phasecounts[x].div(phasecounts['sum'], axis=0)
phaseprops=phasecounts
phaseprops = phaseprops.iloc[:, :5]
phaseprops

In [ ]:
celltypes= ['NSC','Ret-Prog','Cranial-Mes']
for celltype in celltypes:
    test = phaseprops[phaseprops['CellType']==(celltype)]
    test['CellType'] = test['CellType'].cat.remove_unused_categories()
    test.plot.bar(x = 'genotype',stacked=True,title=celltype)
    plt.show()
